## Example 1: msal_extensions functionality test 1

In [ ]:
import msal_extensions
import tempfile
import os
import json

# Example 1: Basic token cache persistence and module overview
print("msal_extensions module imported successfully")

try:
    print("Example 1 - Basic token cache persistence and module overview")
    
    # Display available components
    print("Available msal_extensions components:")
    components = [attr for attr in dir(msal_extensions) if not attr.startswith('_')]
    
    key_components = {
        'PersistedTokenCache': 'Main token cache with persistence',
        'FilePersistence': 'File-based token storage', 
        'FilePersistenceWithDataProtection': 'Encrypted file persistence',
        'KeychainPersistence': 'macOS Keychain storage',
        'LibsecretPersistence': 'Linux libsecret storage',
        'CrossPlatLock': 'Cross-platform file locking',
        'build_encrypted_persistence': 'Factory for encrypted persistence'
    }
    
    for component, description in key_components.items():
        if hasattr(msal_extensions, component):
            print(f"  ✓ {component}: {description}")
        else:
            print(f"  ✗ {component}: Not available")
    
    # Demonstrate basic FilePersistence
    print(f"\n--- File Persistence Demo ---")
    
    # Create temporary file for token cache
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.cache') as temp_file:
        cache_file_path = temp_file.name
        temp_file.write('{}')  # Initialize with empty JSON
    
    print(f"Created temporary cache file: {os.path.basename(cache_file_path)}")
    
    # Create FilePersistence instance
    file_persistence = msal_extensions.FilePersistence(cache_file_path)
    print(f"FilePersistence created successfully")
    print(f"  Cache file: {cache_file_path}")
    print(f"  File exists: {os.path.exists(cache_file_path)}")
    print(f"  File size: {os.path.getsize(cache_file_path)} bytes")
    
    # Test basic persistence operations
    print(f"\nTesting persistence operations:")
    
    # Load initial data
    initial_data = file_persistence.load()
    print(f"  Initial data loaded: {type(initial_data)} ({len(str(initial_data))} chars)")
    
    # Save test data (a persistence stores a STRING; msal's token cache hands it serialized JSON)
    test_data = {'test_token': 'sample_token_value', 'expiry': '2024-12-31T23:59:59'}
    file_persistence.save(json.dumps(test_data))
    print(f"  Test data saved: {len(str(test_data))} chars")
    
    # Reload and verify
    reloaded_data = json.loads(file_persistence.load())
    print(f"  Data reloaded: {type(reloaded_data)}")
    print(f"  Data integrity: {test_data == reloaded_data}")
    assert reloaded_data == test_data, reloaded_data
    
    # Check file size after save
    new_size = os.path.getsize(cache_file_path)
    print(f"  File size after save: {new_size} bytes")
    
    # Cross-platform locking mechanism
    print(f"\n--- Cross-Platform Locking ---")
    
    try:
        # Test CrossPlatLock if available
        if hasattr(msal_extensions, 'CrossPlatLock'):
            lock_file_path = cache_file_path + '.lock'
            
            cross_plat_lock = msal_extensions.CrossPlatLock(lock_file_path)
            print(f"CrossPlatLock created for: {os.path.basename(lock_file_path)}")
            print(f"  Lock file path: {lock_file_path}")
            
            # Test lock operations (non-blocking for demo)
            print(f"  Lock mechanism: Available")
            print(f"  Supports concurrent access protection: Yes")
        else:
            print(f"CrossPlatLock not available in this environment")
    
    except Exception as e:
        print(f"CrossPlatLock error: {e}")
        raise
    
    # Token cache structure analysis
    print(f"\n--- Token Cache Structure ---")
    
    # Demonstrate PersistedTokenCache structure
    try:
        if hasattr(msal_extensions, 'PersistedTokenCache'):
            # Create a basic persisted token cache
            persisted_cache = msal_extensions.PersistedTokenCache(file_persistence)
            print(f"PersistedTokenCache created successfully")
            print(f"  Type: {type(persisted_cache)}")
            
            # Check if it has expected methods
            cache_methods = ['add', 'find', 'update', 'remove']
            available_methods = [method for method in cache_methods if hasattr(persisted_cache, method)]
            print(f"  Available cache methods: {available_methods}")
            
            # Check serialization methods
            serialization_methods = ['serialize', 'deserialize']
            available_ser_methods = [method for method in serialization_methods if hasattr(persisted_cache, method)]
            print(f"  Serialization methods: {available_ser_methods}")
        else:
            print(f"PersistedTokenCache not available")
    
    except Exception as e:
        print(f"PersistedTokenCache error: {e}")
        raise
    
    # Platform-specific persistence options
    print(f"\n--- Platform-Specific Options ---")
    
    platform_options = [
        ('KeychainPersistence', 'macOS Keychain'),
        ('LibsecretPersistence', 'Linux libsecret'),
        ('FilePersistenceWithDataProtection', 'Windows DPAPI')
    ]
    
    for persistence_class, platform_desc in platform_options:
        if hasattr(msal_extensions, persistence_class):
            print(f"  ✓ {persistence_class}: {platform_desc} - Available")
        else:
            print(f"  ✗ {persistence_class}: {platform_desc} - Not available")
    
    # Cleanup
    try:
        os.unlink(cache_file_path)
        lock_file = cache_file_path + '.lock'
        if os.path.exists(lock_file):
            os.unlink(lock_file)
        print(f"\n✓ Cleanup completed")
    except:
        pass
        raise
    
    print("msal_extensions example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal_extensions example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: msal_extensions functionality test 2

In [ ]:
import msal_extensions
import tempfile
import os
import json

# Example 2: Encrypted persistence and secure token storage
print("msal_extensions encrypted persistence and secure token storage")

try:
    print("Example 2 - Encrypted persistence and secure token storage")
    
    # Test encrypted persistence factory
    print("--- Encrypted Persistence Factory ---")
    
    if hasattr(msal_extensions, 'build_encrypted_persistence'):
        print("build_encrypted_persistence function available")
        
        # Create temporary cache location
        with tempfile.TemporaryDirectory() as temp_dir:
            cache_location = os.path.join(temp_dir, 'encrypted_cache.bin')
            
            try:
                encrypted_persistence = msal_extensions.build_encrypted_persistence(cache_location)
                is_encrypted = True
            except Exception as e:
                # The OS secret store may be missing or unusable here (e.g. no PyGObject or no
                # running Secret Service on a headless Linux box): fall back to a plain file.
                print(f"  Encrypted persistence unavailable ({type(e).__name__}: {e}); falling back to FilePersistence")
                encrypted_persistence = msal_extensions.FilePersistence(cache_location)
                is_encrypted = False
            
            print(f"Encrypted persistence created:")
            print(f"  Type: {type(encrypted_persistence)}")
            print(f"  Cache location: {os.path.basename(cache_location)}")
            
            # Test encrypted save/load operations
            print(f"\nTesting encrypted operations:")
            
            # Test data representing token information
            sensitive_data = {
                'access_token': 'eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsImtpZ...',
                'refresh_token': 'OAAABAAAAiL9Kn2Z27UubvWFPbm0gLWQq6_...',
                'expires_in': 3600,
                'token_type': 'Bearer',
                'scope': 'https://graph.microsoft.com/.default',
                'client_id': '12345678-1234-1234-1234-123456789012',
                'account': {
                    'username': 'user@example.com',
                    'name': 'Test User',
                    'home_account_id': 'abcd1234.efgh5678'
                }
            }
            
            print(f"  Original data size: {len(json.dumps(sensitive_data))} chars")
            
            # Save encrypted data (a persistence stores a STRING: serialize the token record first)
            encrypted_persistence.save(json.dumps(sensitive_data))
            print(f"  Data saved with encryption")
            
            # Check if file was created and is not plain text
            if os.path.exists(cache_location):
                file_size = os.path.getsize(cache_location)
                print(f"  Encrypted file size: {file_size} bytes")
                
                # Read raw file content to verify encryption
                with open(cache_location, 'rb') as f:
                    raw_content = f.read()
                
                # Check if content looks encrypted (not plain JSON)
                try:
                    json.loads(raw_content.decode('utf-8'))
                    print(f"  ⚠️  File appears to be plain text")
                    assert not is_encrypted, "an encrypted persistence wrote plaintext JSON"
                except (json.JSONDecodeError, UnicodeDecodeError):
                    print(f"  ✓ File content is encrypted/binary")
            
            # Load and verify encrypted data
            loaded_data = json.loads(encrypted_persistence.load())
            print(f"  Data loaded from encryption")
            print(f"  Loaded data type: {type(loaded_data)}")
            
            # Verify data integrity
            if isinstance(loaded_data, dict) and 'access_token' in loaded_data:
                print(f"  ✓ Token structure preserved")
                print(f"  ✓ Access token length: {len(loaded_data['access_token'])} chars")
                print(f"  ✓ Data integrity: {loaded_data == sensitive_data}")
            assert loaded_data == sensitive_data, loaded_data
            
    else:
        print("build_encrypted_persistence not available")
    
    # Test platform-specific secure storage
    print(f"\n--- Platform-Specific Secure Storage ---")
    
    # Test different persistence backends
    persistence_backends = [
        ('FilePersistence', 'Plain file storage'),
        ('FilePersistenceWithDataProtection', 'Windows DPAPI encryption'),
        ('KeychainPersistence', 'macOS Keychain'),
        ('LibsecretPersistence', 'Linux libsecret')
    ]
    
    available_backends = []
    
    for backend_name, description in persistence_backends:
        if hasattr(msal_extensions, backend_name):
            available_backends.append((backend_name, description))
            print(f"  ✓ {backend_name}: {description}")
        else:
            print(f"  ✗ {backend_name}: {description} (not available)")
    
    # Demonstrate secure storage simulation
    print(f"\n--- Secure Storage Simulation ---")
    
    class TokenSecurityAnalyzer:
        def __init__(self):
            self.security_levels = {
                'FilePersistence': 1,  # Plain file
                'FilePersistenceWithDataProtection': 3,  # OS-level encryption
                'KeychainPersistence': 4,  # OS keychain
                'LibsecretPersistence': 4   # Linux secure storage
            }
        
        def analyze_security_level(self, persistence_type):
            level = self.security_levels.get(persistence_type, 0)
            
            descriptions = {
                0: 'Unknown security level',
                1: 'Basic - File system permissions only',
                2: 'Moderate - Application-level encryption',
                3: 'Good - OS-level encryption (DPAPI)',
                4: 'Excellent - OS secure storage (Keychain/libsecret)'
            }
            
            return level, descriptions.get(level, 'Unknown')
        
        def recommend_persistence(self, platform):
            recommendations = {
                'Windows': 'FilePersistenceWithDataProtection',
                'Darwin': 'KeychainPersistence',  # macOS
                'Linux': 'LibsecretPersistence',
                'default': 'build_encrypted_persistence'
            }
            
            return recommendations.get(platform, recommendations['default'])
    
    analyzer = TokenSecurityAnalyzer()
    
    print("Security analysis of available persistence backends:")
    for backend_name, description in available_backends:
        level, level_desc = analyzer.analyze_security_level(backend_name)
        print(f"  {backend_name}:")
        print(f"    Security level: {level}/4 - {level_desc}")
        print(f"    Use case: {description}")
    
    # Token cache security best practices
    print(f"\n--- Security Best Practices ---")
    
    security_practices = [
        "Use platform-specific secure storage when available",
        "Encrypt cache files with build_encrypted_persistence",
        "Set appropriate file system permissions (600)",
        "Use file locking to prevent concurrent access issues",
        "Regularly rotate and refresh cached tokens",
        "Clear sensitive data from memory after use",
        "Monitor cache file access and modifications",
        "Implement cache expiration and cleanup policies"
    ]
    
    print("Token cache security recommendations:")
    for i, practice in enumerate(security_practices, 1):
        print(f"  {i}. {practice}")
    
    # Cache lifecycle management
    print(f"\n--- Cache Lifecycle Management ---")
    
    cache_lifecycle = {
        'Initialization': [
            'Choose appropriate persistence backend',
            'Create cache directory with secure permissions',
            'Initialize empty cache structure'
        ],
        'Token Storage': [
            'Encrypt sensitive token data',
            'Store with expiration timestamps',
            'Maintain cache coherency with locks'
        ],
        'Token Retrieval': [
            'Validate token expiration',
            'Decrypt and return valid tokens',
            'Handle cache misses gracefully'
        ],
        'Cache Maintenance': [
            'Remove expired tokens automatically',
            'Compact cache file periodically',
            'Monitor cache size and performance'
        ],
        'Cleanup': [
            'Securely delete cache files',
            'Clear sensitive data from memory',
            'Remove lock files and temporary data'
        ]
    }
    
    print("Token cache lifecycle stages:")
    for stage, tasks in cache_lifecycle.items():
        print(f"\n{stage}:")
        for task in tasks:
            print(f"  • {task}")
    
    print("msal_extensions example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal_extensions example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: msal_extensions functionality test 3

In [ ]:
import msal_extensions
import tempfile
import threading
import time
import json
import os

# Example 3: Concurrent access and file locking mechanisms
print("msal_extensions concurrent access and file locking mechanisms")

try:
    print("Example 3 - Concurrent access and file locking mechanisms")
    
    # File locking demonstration
    print("--- File Locking Demonstration ---")
    
    class ConcurrentCacheAccessSimulator:
        def __init__(self):
            self.results = []
            self.access_attempts = []
        
        def simulate_cache_access(self, thread_id, cache_file, lock_file, duration=0.5):
            """Simulate concurrent cache access with locking"""
            try:
                print(f"  Thread {thread_id}: Attempting to access cache")
                
                # Create lock
                if hasattr(msal_extensions, 'CrossPlatLock'):
                    lock = msal_extensions.CrossPlatLock(lock_file)
                    print(f"  Thread {thread_id}: Lock created")
                    
                    # Simulate lock acquisition
                    start_time = time.time()
                    
                    # In a real scenario, lock would be acquired here
                    # For demo purposes, we'll simulate the operation
                    print(f"  Thread {thread_id}: Acquiring lock...")
                    time.sleep(0.1)  # Simulate lock acquisition time
                    
                    print(f"  Thread {thread_id}: Lock acquired, performing cache operations")
                    
                    # Simulate cache operations
                    time.sleep(duration)
                    
                    end_time = time.time()
                    elapsed = end_time - start_time
                    
                    result = {
                        'thread_id': thread_id,
                        'start_time': start_time,
                        'end_time': end_time,
                        'elapsed_time': elapsed,
                        'status': 'completed'
                    }
                    
                    self.results.append(result)
                    print(f"  Thread {thread_id}: Operations completed in {elapsed:.3f}s")
                    
                else:
                    print(f"  Thread {thread_id}: CrossPlatLock not available")
                    
            except Exception as e:
                error_result = {
                    'thread_id': thread_id,
                    'status': 'error',
                    'error': str(e)
                }
                self.results.append(error_result)
                print(f"  Thread {thread_id}: Error - {e}")
        
        def get_statistics(self):
            if not self.results:
                return {'message': 'No results available'}
            
            completed_results = [r for r in self.results if r.get('status') == 'completed']
            error_results = [r for r in self.results if r.get('status') == 'error']
            
            stats = {
                'total_threads': len(self.results),
                'completed': len(completed_results),
                'errors': len(error_results)
            }
            
            if completed_results:
                elapsed_times = [r['elapsed_time'] for r in completed_results]
                stats.update({
                    'avg_time': sum(elapsed_times) / len(elapsed_times),
                    'min_time': min(elapsed_times),
                    'max_time': max(elapsed_times)
                })
            
            return stats
    
    # Create temporary files for testing
    with tempfile.TemporaryDirectory() as temp_dir:
        cache_file = os.path.join(temp_dir, 'test_cache.json')
        lock_file = os.path.join(temp_dir, 'test_cache.lock')
        
        # Initialize cache file
        with open(cache_file, 'w') as f:
            json.dump({}, f)
        
        simulator = ConcurrentCacheAccessSimulator()
        
        # Create multiple threads to simulate concurrent access
        threads = []
        num_threads = 3
        
        print(f"Starting {num_threads} threads to simulate concurrent access:")
        
        for i in range(num_threads):
            thread = threading.Thread(
                target=simulator.simulate_cache_access,
                args=(i+1, cache_file, lock_file, 0.2 + i*0.1)
            )
            threads.append(thread)
        
        # Start all threads
        start_time = time.time()
        for thread in threads:
            thread.start()
            time.sleep(0.05)  # Small delay between thread starts
        
        # Wait for all threads to complete
        for thread in threads:
            thread.join()
        
        total_time = time.time() - start_time
        
        print(f"\nConcurrent access simulation completed in {total_time:.3f}s")
        
        # Analyze results
        stats = simulator.get_statistics()
        print(f"\nConcurrent Access Statistics:")
        for key, value in stats.items():
            if key.endswith('_time'):
                print(f"  {key}: {value:.3f}s")
            else:
                print(f"  {key}: {value}")
    
    # Token cache consistency demonstration
    print(f"\n--- Token Cache Consistency ---")
    
    class TokenCacheConsistencyDemo:
        def __init__(self):
            self.cache_operations = []
            self.consistency_checks = []
        
        def simulate_cache_operation(self, operation_type, data=None):
            """Simulate cache operations with consistency tracking"""
            timestamp = time.time()
            
            operation = {
                'type': operation_type,
                'timestamp': timestamp,
                'data_hash': hash(str(data)) if data else None
            }
            
            self.cache_operations.append(operation)
            print(f"  Operation: {operation_type} at {timestamp:.3f}")
            
            return operation
        
        def check_consistency(self):
            """Check cache consistency across operations"""
            if len(self.cache_operations) < 2:
                return True
            
            # Check for overlapping operations that might cause inconsistency
            overlapping_ops = []
            
            for i, op1 in enumerate(self.cache_operations[:-1]):
                for j, op2 in enumerate(self.cache_operations[i+1:], i+1):
                    time_diff = abs(op1['timestamp'] - op2['timestamp'])
                    if time_diff < 0.1:  # Operations within 0.1 seconds
                        overlapping_ops.append((i, j, time_diff))
            
            consistency_result = {
                'total_operations': len(self.cache_operations),
                'overlapping_operations': len(overlapping_ops),
                'is_consistent': len(overlapping_ops) == 0
            }
            
            self.consistency_checks.append(consistency_result)
            return consistency_result
    
    # Demonstrate cache consistency
    consistency_demo = TokenCacheConsistencyDemo()
    
    print("Simulating token cache operations:")
    
    # Simulate various cache operations
    cache_operations = [
        ('READ_TOKEN', {'token_type': 'access_token'}),
        ('WRITE_TOKEN', {'token_type': 'access_token', 'value': 'new_token_123'}),
        ('UPDATE_EXPIRY', {'token_type': 'access_token', 'expires_at': time.time() + 3600}),
        ('READ_TOKEN', {'token_type': 'refresh_token'}),
        ('DELETE_EXPIRED', {'expired_count': 2})
    ]
    
    for op_type, op_data in cache_operations:
        consistency_demo.simulate_cache_operation(op_type, op_data)
        time.sleep(0.05)  # Small delay between operations
    
    # Check consistency
    consistency_result = consistency_demo.check_consistency()
    print(f"\nConsistency Check Results:")
    for key, value in consistency_result.items():
        print(f"  {key}: {value}")
    
    # Performance implications of locking
    print(f"\n--- Performance Analysis ---")
    
    performance_scenarios = [
        {
            'name': 'No Locking',
            'description': 'Direct file access without locking',
            'pros': ['Fastest access', 'No blocking'],
            'cons': ['Race conditions', 'Data corruption risk', 'Inconsistent reads']
        },
        {
            'name': 'File Locking',
            'description': 'CrossPlatLock for cache access',
            'pros': ['Data consistency', 'Prevents corruption', 'Safe concurrent access'],
            'cons': ['Slower access', 'Potential blocking', 'Lock file overhead']
        },
        {
            'name': 'Memory Caching',
            'description': 'In-memory cache with periodic persistence',
            'pros': ['Fast reads', 'Reduced file I/O', 'Better performance'],
            'cons': ['Memory usage', 'Data loss risk', 'Synchronization complexity']
        }
    ]
    
    print("Performance scenario analysis:")
    for scenario in performance_scenarios:
        print(f"\n{scenario['name']}: {scenario['description']}")
        print(f"  Pros: {', '.join(scenario['pros'])}")
        print(f"  Cons: {', '.join(scenario['cons'])}")
    
    # Best practices for concurrent access
    print(f"\n--- Concurrent Access Best Practices ---")
    
    best_practices = [
        "Always use file locking for cache modifications",
        "Keep lock duration as short as possible",
        "Implement timeout mechanisms for lock acquisition",
        "Use read-write locks when available (separate read/write access)",
        "Consider cache partitioning for high-concurrency scenarios",
        "Implement retry logic with exponential backoff",
        "Monitor lock contention and adjust cache strategy",
        "Use atomic operations where possible"
    ]
    
    print("Concurrent access best practices:")
    for i, practice in enumerate(best_practices, 1):
        print(f"  {i}. {practice}")
    
    print("msal_extensions example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal_extensions example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: msal_extensions functionality test 4

In [ ]:
import msal_extensions
import tempfile
import os

# Example 4: Integration with MSAL library and token management workflows
print("msal_extensions integration with MSAL library and token management workflows")

try:
    print("Example 4 - Integration with MSAL library and token management workflows")
    
    # MSAL integration simulation
    print("--- MSAL Integration Simulation ---")
    
    class MSALTokenCacheIntegration:
        def __init__(self):
            self.cache_stats = {
                'cache_hits': 0,
                'cache_misses': 0,
                'tokens_stored': 0,
                'tokens_retrieved': 0,
                'cache_size': 0
            }
        
        def create_token_cache_with_persistence(self, cache_location):
            """Create a persistent token cache for MSAL integration"""
            try:
                # Create persistence layer
                persistence = msal_extensions.FilePersistence(cache_location)
                print(f"File persistence created: {os.path.basename(cache_location)}")
                
                # Create persistent token cache
                token_cache = msal_extensions.PersistedTokenCache(persistence)
                print(f"Persistent token cache created: {type(token_cache)}")
                
                return token_cache, persistence
            
            except Exception as e:
                print(f"Error creating token cache: {e}")
                return None, None
        
        def simulate_msal_workflow(self, token_cache):
            """Simulate typical MSAL authentication workflow"""
            print(f"\nSimulating MSAL authentication workflow:")
            
            # Simulate token acquisition scenarios
            scenarios = [
                {
                    'name': 'Initial Authentication',
                    'description': 'First-time login with username/password',
                    'tokens': {
                        'access_token': 'eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIs...',
                        'refresh_token': 'OAAABAAAAiL9Kn2Z27UubvWFPbm0gL...',
                        'id_token': 'eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIs...',
                        'expires_in': 3600,
                        'scope': 'https://graph.microsoft.com/.default'
                    }
                },
                {
                    'name': 'Silent Token Acquisition',
                    'description': 'Getting token from cache without user interaction',
                    'tokens': {
                        'access_token': 'eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIs...',
                        'expires_in': 3600,
                        'scope': 'https://graph.microsoft.com/.default'
                    }
                },
                {
                    'name': 'Token Refresh',
                    'description': 'Using refresh token to get new access token',
                    'tokens': {
                        'access_token': 'eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIs...',
                        'refresh_token': 'OAAABAAAAiL9Kn2Z27UubvWFPbm0gL...',
                        'expires_in': 3600
                    }
                }
            ]
            
            for scenario in scenarios:
                print(f"\n  Scenario: {scenario['name']}")
                print(f"  Description: {scenario['description']}")
                
                # Simulate storing tokens in cache
                if token_cache:
                    try:
                        # In real MSAL, this would be done internally
                        # Here we simulate the process
                        token_count = len(scenario['tokens'])
                        self.cache_stats['tokens_stored'] += token_count
                        
                        print(f"    Tokens stored in cache: {token_count}")
                        print(f"    Token types: {list(scenario['tokens'].keys())}")
                        
                        # Simulate cache size calculation
                        self.cache_stats['cache_size'] += sum(len(str(v)) for v in scenario['tokens'].values())
                        
                    except Exception as e:
                        print(f"    Error storing tokens: {e}")
        
        def simulate_cache_operations(self, token_cache):
            """Simulate various cache operations"""
            print(f"\n--- Cache Operations Simulation ---")
            
            operations = [
                ('LOOKUP_BY_SCOPE', 'https://graph.microsoft.com/.default'),
                ('LOOKUP_BY_CLIENT_ID', '12345678-1234-1234-1234-123456789012'),
                ('LOOKUP_BY_USER', 'user@example.com'),
                ('CLEANUP_EXPIRED', None),
                ('GET_ACCOUNTS', None)
            ]
            
            for operation, parameter in operations:
                print(f"  Operation: {operation}")
                if parameter:
                    print(f"    Parameter: {parameter}")
                
                # Simulate operation results
                if operation == 'LOOKUP_BY_SCOPE':
                    self.cache_stats['cache_hits'] += 1
                    print(f"    Result: Token found in cache")
                elif operation == 'LOOKUP_BY_CLIENT_ID':
                    self.cache_stats['cache_hits'] += 1
                    print(f"    Result: 3 tokens found for client")
                elif operation == 'LOOKUP_BY_USER':
                    self.cache_stats['cache_misses'] += 1
                    print(f"    Result: No tokens found for user")
                elif operation == 'CLEANUP_EXPIRED':
                    print(f"    Result: 2 expired tokens removed")
                elif operation == 'GET_ACCOUNTS':
                    print(f"    Result: 1 account found in cache")
        
        def get_integration_statistics(self):
            return self.cache_stats
    
    # Create MSAL integration instance
    msal_integration = MSALTokenCacheIntegration()
    
    # Create persistent cache
    with tempfile.TemporaryDirectory() as temp_dir:
        cache_location = os.path.join(temp_dir, 'msal_token_cache.bin')
        
        token_cache, persistence = msal_integration.create_token_cache_with_persistence(cache_location)
        
        if token_cache:
            # Simulate MSAL workflow
            msal_integration.simulate_msal_workflow(token_cache)
            
            # Simulate cache operations
            msal_integration.simulate_cache_operations(token_cache)
            
            # Show integration statistics
            stats = msal_integration.get_integration_statistics()
            print(f"\nIntegration Statistics:")
            for key, value in stats.items():
                print(f"  {key}: {value}")
    
    # Azure AD application integration patterns
    print(f"\n--- Azure AD Application Patterns ---")
    
    application_patterns = {
        'Web Application': {
            'flow': 'Authorization Code',
            'cache_strategy': 'Server-side persistence',
            'token_types': ['access_token', 'refresh_token', 'id_token'],
            'persistence': 'Database or encrypted file',
            'considerations': ['Session management', 'CSRF protection', 'Secure storage']
        },
        'Desktop Application': {
            'flow': 'Authorization Code with PKCE',
            'cache_strategy': 'Local encrypted cache',
            'token_types': ['access_token', 'refresh_token'],
            'persistence': 'OS-specific secure storage',
            'considerations': ['Cross-platform compatibility', 'User data protection', 'Silent refresh']
        },
        'Mobile Application': {
            'flow': 'Authorization Code with PKCE',
            'cache_strategy': 'Device keychain/keystore',
            'token_types': ['access_token', 'refresh_token'],
            'persistence': 'Platform secure storage',
            'considerations': ['App lifecycle', 'Biometric authentication', 'Deep linking']
        },
        'Service/Daemon': {
            'flow': 'Client Credentials',
            'cache_strategy': 'Memory or secure file',
            'token_types': ['access_token'],
            'persistence': 'Optional (tokens are app-scoped)',
            'considerations': ['Certificate management', 'Token refresh', 'High availability']
        }
    }
    
    print("Azure AD application integration patterns:")
    for app_type, config in application_patterns.items():
        print(f"\n{app_type}:")
        for key, value in config.items():
            if isinstance(value, list):
                print(f"  {key}: {', '.join(value)}")
            else:
                print(f"  {key}: {value}")
    
    # Token cache configuration recommendations
    print(f"\n--- Cache Configuration Recommendations ---")
    
    cache_configurations = {
        'Development': {
            'persistence': 'FilePersistence',
            'location': './dev_cache.json',
            'encryption': False,
            'cleanup_interval': '1 hour',
            'max_cache_size': '1 MB'
        },
        'Production Web App': {
            'persistence': 'Database + FilePersistenceWithDataProtection',
            'location': '/secure/cache/tokens.enc',
            'encryption': True,
            'cleanup_interval': '15 minutes',
            'max_cache_size': '10 MB'
        },
        'Production Desktop': {
            'persistence': 'KeychainPersistence (macOS) / CredentialManager (Windows)',
            'location': 'OS-managed',
            'encryption': True,
            'cleanup_interval': '30 minutes',
            'max_cache_size': '5 MB'
        },
        'High-Security Environment': {
            'persistence': 'Hardware Security Module (HSM)',
            'location': 'HSM-managed',
            'encryption': True,
            'cleanup_interval': '5 minutes',
            'max_cache_size': '1 MB'
        }
    }
    
    print("Cache configuration by environment:")
    for env, config in cache_configurations.items():
        print(f"\n{env}:")
        for setting, value in config.items():
            print(f"  {setting}: {value}")
    
    # Token lifecycle management
    print(f"\n--- Token Lifecycle Management ---")
    
    lifecycle_stages = {
        'Acquisition': [
            'User authentication (interactive or silent)',
            'Token request to Azure AD',
            'Token validation and parsing',
            'Secure storage in persistent cache'
        ],
        'Usage': [
            'Token retrieval from cache',
            'Expiration check before use',
            'Automatic refresh if needed',
            'Secure transmission to resource'
        ],
        'Refresh': [
            'Detect token expiration',
            'Use refresh token for new access token',
            'Update cache with new tokens',
            'Handle refresh token rotation'
        ],
        'Cleanup': [
            'Periodic removal of expired tokens',
            'Account-based token removal',
            'Cache size management',
            'Secure deletion of sensitive data'
        ]
    }
    
    print("Token lifecycle management stages:")
    for stage, tasks in lifecycle_stages.items():
        print(f"\n{stage}:")
        for task in tasks:
            print(f"  • {task}")
    
    print("msal_extensions example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal_extensions example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: msal_extensions functionality test 5

In [ ]:
import msal_extensions

# Example 5: Test msal_extensions basic functionality
print(f"msal_extensions module imported successfully")

try:
    # Basic package operations
    module = msal_extensions
    print(f"Example 5 - msal_extensions working")
    print(f"Module type: {type(module).__name__}")
    
    # Test module attributes
    attrs = [attr for attr in dir(module) if not attr.startswith('_')]
    print(f"Public attributes: {len(attrs)}")
    
    # Show some key attributes if available
    if len(attrs) > 0:
        print(f"Sample attributes: {attrs[:5]}")
    
    print(f"msal_extensions example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal_extensions example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print(f"Example 5 finished")